<a href="https://colab.research.google.com/github/CuriousTechNomad/slm-agentic-software-engineering/blob/main/notebooks/05_trace_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1
Markdown

↓

Cell 2
Project Setup

↓

Cell 3
Imports

↓

Cell 4
Load Agent Traces

↓

Cell 5
Inspect One Trace

↓

Cell 6
Convert Trace → Training Example

↓

Cell 7
Generate Training Dataset

↓

Cell 8
Inspect Dataset

↓

Cell 9
Save JSONL

↓

Cell 10
Dataset Statistics

↓

Cell 11
Save Metadata

# Notebook 05 - Agent Trace Dataset Generation

## Objective

Convert multi-agent reasoning traces into a supervised fine-tuning dataset.

Input

- agent_traces.json

Output

- training_dataset.jsonl

This dataset will be used in Notebook 06 for local SLM fine-tuning.

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

PROJECT_DIR = Path("/content/drive/MyDrive/LLM_Project")

OUTPUT_DIR = PROJECT_DIR / "outputs"
DATA_DIR = PROJECT_DIR / "data"

Mounted at /content/drive


In [2]:
import json

import pandas as pd

from pathlib import Path

In [3]:
with open(
    OUTPUT_DIR / "agent_traces.json"
) as f:

    traces = json.load(f)

print("Loaded traces:", len(traces))

Loaded traces: 1


In [4]:
sample = traces[0]

print(sample.keys())

dict_keys(['instance_id', 'repo', 'problem_statement', 'steps'])


In [5]:
def create_training_example(trace):

    reasoning = ""

    for step in trace["steps"]:

        reasoning += f"""

### {step["agent"]}

{step["response"]}

"""

    example = {

        "instruction":

        "Resolve the following GitHub issue using systematic software engineering reasoning.",

        "input":

        trace["problem_statement"],

        "output":

        reasoning.strip()

    }

    return example

In [6]:
dataset = []

for trace in traces:

    dataset.append(

        create_training_example(trace)

    )

print("Training Examples:", len(dataset))

Training Examples: 1


In [7]:
print(dataset[0]["instruction"])

print("="*80)

print(dataset[0]["input"][:500])

print("="*80)

print(dataset[0]["output"][:1000])

Resolve the following GitHub issue using systematic software engineering reasoning.
Raise error when blueprint name contains a dot
This is required since every dot is now significant since blueprints can be nested. An error was already added for endpoint names in 1.0, but should have been added for this as well.

### Task Planner

Investigation Plan:

1. Verify if the issue is reproducible on the latest version of Flask (1.x series).
2. Check if the issue occurs with different Python versions and operating systems.
3. Review the code changes introduced in Flask 1.0 that specifically address endpoint names.
4. Investigate if there are any specific configurations or settings that might affect the behavior when using blueprints with dots in their names.
5. Examine the error message provided in the GitHub issue to understand the exact nature of the error being raised.
6. Search for similar issues or discussions related to this problem



### Repository Investigator

Response:
1. Likely Fil

In [8]:
jsonl_path = OUTPUT_DIR / "training_dataset.jsonl"

with open(
    jsonl_path,
    "w"
) as f:

    for row in dataset:

        f.write(json.dumps(row))

        f.write("\n")

print(jsonl_path)

/content/drive/MyDrive/LLM_Project/outputs/training_dataset.jsonl


In [9]:
stats = {

    "examples": len(dataset),

    "avg_instruction_length":

    sum(
        len(x["instruction"])
        for x in dataset
    ) / len(dataset),

    "avg_input_length":

    sum(
        len(x["input"])
        for x in dataset
    ) / len(dataset),

    "avg_output_length":

    sum(
        len(x["output"])
        for x in dataset
    ) / len(dataset)

}

stats

{'examples': 1,
 'avg_instruction_length': 83.0,
 'avg_input_length': 230.0,
 'avg_output_length': 3345.0}

In [12]:
with open(
    OUTPUT_DIR / "training_metadata.json",
    "w"
) as f:

    json.dump(
        stats,
        f,
        indent=4
    )

print("Metadata Saved")

Metadata Saved


outputs/

training_dataset.jsonl

training_metadata.json